# 5. Join your data by edge_id

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/05_join_by_edge_id.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/05_join_by_edge_id.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

You join traffic counts to the edges by `edge_id` and color the map by the counts. The counts in this notebook are made up.

`edge_id` is a `BIGINT` content hash of the OSM way and the two ends of the edge. It is the same after a rebuild, so data joined on it stays correct.

In [ ]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
try:
    import duckosm
except ImportError:
    %pip install -q "duckosm[widget]" roadstyle
    import duckosm
print("duckosm", duckosm.__version__)

In [ ]:
area = duckosm.get("Monaco", out="data")      # the OSM file of Monaco and its boundary
net = duckosm.build(pbf=area.pbf, boundary=area.boundary, modes=["driving"],
                    output="data/monaco.duckdb")   # the driving network: a duckosm.Network
net

## Make the counts

One count for each direction of the main roads, as an external file can give them. Keep the column type `int64`: a float or a text column loses the hash.

In [ ]:
import numpy as np
from IPython.display import HTML    # to show a table with no index column and no NaN

main = net.edges(where="highway IN ('primary', 'secondary', 'tertiary')")       # the main roads
base = main.highway.map({"primary": 900, "secondary": 500, "tertiary": 250})    # vehicles per hour, by class
rng = np.random.default_rng(1)
counts = main[["edge_id"]].assign(vehicles_per_hour=(base * rng.uniform(0.5, 1.5, len(main))).round().astype(int))
print(counts.dtypes.to_dict())
HTML(counts.head().to_html(index=False))

## Join on edge_id

pandas joins the counts to the edges on the column `edge_id`. Edges without a count get no value.

In [ ]:
gdf = net.edges().merge(counts, on="edge_id", how="left")    # the edges with their counts
print(len(counts), "counts,", gdf.vehicles_per_hour.notna().sum(), "joined")
HTML(gdf.nlargest(5, "vehicles_per_hour")[["edge_id", "name", "highway", "vehicles_per_hour"]].to_html(index=False, na_rep=""))

## Color the map by the counts

Edges without a count are gray. Run the cell to see the live map: you can zoom, and click a road to see its columns. This page shows a picture of the map.

In [ ]:
net.map(data=gdf, color_by="vehicles_per_hour", cmap="magma_r")    # a map, colored by the count

## The ids stay after a rebuild

Build Monaco again into a new file. The same counts join to the same edges, with no change.

In [ ]:
again = duckosm.build(pbf=area.pbf, boundary=area.boundary, modes=["driving"], output="data/monaco_again.duckdb")
n = counts.edge_id.isin(again.edges().edge_id).sum()    # the counts whose edge_id is in the new build
print(n, "of", len(counts), "counts join to the new build")